# 09 — Crash Archetypes using K-Means Clustering

## Objective

Identify natural groups (**crash archetypes**) with similar characteristics using unsupervised K-Means clustering, following the lecturer's K-Means practical workflow.

### Workflow

1. Prepare numerical features
2. Impute missing values
3. Standardize features
4. Apply **K-Means with K-Means++ initialization**
5. Compare **K = 2 through 10**
6. Evaluate clustering using **WCSS/Inertia** and **Silhouette Score**
7. Inspect the **Elbow Curve**
8. Select a final K
9. Profile the resulting crash archetypes
10. Examine their severity composition **after** clustering
11. Assign the learned clusters to the unseen test set

**Important:** `SEVERITY` is never used to fit the clusters. It is used only after clustering to interpret the discovered archetypes.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("Libraries imported successfully.")

In [ ]:
DATA_DIR = "../data"

X_train_full = sparse.load_npz(os.path.join(DATA_DIR, "X_train_final.npz"))
X_test_full = sparse.load_npz(os.path.join(DATA_DIR, "X_test_final.npz"))

y_train = np.load(os.path.join(DATA_DIR, "y_train_encoded.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test_encoded.npy"))

feature_names = pd.read_csv(
    os.path.join(DATA_DIR, "final_feature_names.csv")
)["feature_name"].tolist()

selected_original = pd.read_csv(
    os.path.join(DATA_DIR, "selected_features_ig.csv")
)["original_feature"].tolist()

print("Full training matrix :", X_train_full.shape)
print("Full testing matrix  :", X_test_full.shape)
print("Selected features    :", len(selected_original))

In [ ]:
# Reproduce the exact 26-original-feature -> 136-encoded-feature mapping.
selected_indices = []

for original_feature in selected_original:
    exact_matches = [
        i for i, name in enumerate(feature_names)
        if name == original_feature
    ]

    encoded_matches = [
        i for i, name in enumerate(feature_names)
        if name.startswith(original_feature + "_")
    ]

    selected_indices.extend(exact_matches)
    selected_indices.extend(encoded_matches)

selected_indices = list(dict.fromkeys(selected_indices))

X_train_selected = X_train_full[:, selected_indices]
X_test_selected = X_test_full[:, selected_indices]

selected_feature_names = [
    feature_names[i] for i in selected_indices
]

print("Selected original features :", len(selected_original))
print("Selected encoded features  :", len(selected_indices))
print("Training matrix             :", X_train_selected.shape)
print("Testing matrix              :", X_test_selected.shape)

In [ ]:
# The preprocessing pipeline already standardized the numerical representation.
# K-Means therefore uses the final selected standardized feature space directly.

X_train_cluster = X_train_selected.toarray().astype(np.float32)

print("Clustering matrix:", X_train_cluster.shape)
print("Data type:", X_train_cluster.dtype)

## K-Means Concepts from the Lecturer Practical

### Lloyd's algorithm

K-Means follows an iterative **assignment → update** procedure:

1. Initialize K centroids.
2. Assign each observation to the nearest centroid.
3. Recalculate each centroid as the mean of the observations assigned to it.
4. Repeat until the assignments/centroids stabilize or the iteration limit is reached.

### K-Means++

K-Means++ provides a more informed initialization of the centroids than purely random initialization. It is used here through `init="k-means++"`.

### Why scaling matters

K-Means is distance-based. Standardization prevents variables with larger numerical scales from dominating the distance calculation.

### Cluster-quality measures

- **WCSS / Inertia:** measures the within-cluster squared distances. Lower values indicate more compact clusters, although inertia naturally decreases as K increases.
- **Elbow method:** looks for a point where additional clusters provide diminishing reductions in WCSS.
- **Silhouette Score:** considers both within-cluster cohesion and separation from other clusters. Higher values indicate better-defined clusters.

There may not always be a single obvious elbow, so K should be selected using the evidence from both diagnostics and the interpretability of the resulting clusters.

## Choosing the Number of Clusters

Different values of K are compared using:

- **WCSS (Within-Cluster Sum of Squares):** measures cluster compactness; lower is better for a fixed K.
- **Silhouette Score:** measures cohesion and separation; higher values indicate better-separated clusters.

K-Means++ is used for centroid initialization, followed by the standard Lloyd-style Assign → Update → Repeat procedure.

In [ ]:
# Compare K = 2 through 10.
# WCSS uses the complete training set.
# Silhouette uses a reproducible sample to keep runtime practical.

K_VALUES = range(2, 11)
SAMPLE_SIZE = min(10000, len(X_train_cluster))

rng = np.random.default_rng(42)
silhouette_indices = rng.choice(
    len(X_train_cluster),
    size=SAMPLE_SIZE,
    replace=False
)
X_silhouette = X_train_cluster[silhouette_indices]

clustering_results = []

for k in K_VALUES:
    print(f"\nEvaluating K = {k} ...")

    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=42,
    )

    labels = kmeans.fit_predict(X_train_cluster)

    silhouette = silhouette_score(
        X_silhouette,
        labels[silhouette_indices],
        metric="euclidean"
    )

    clustering_results.append({
        "K": k,
        "WCSS": kmeans.inertia_,
        "Silhouette": silhouette,
    })

    print(f"WCSS       : {kmeans.inertia_:,.2f}")
    print(f"Silhouette : {silhouette:.4f}")

clustering_results_df = pd.DataFrame(clustering_results)
display(clustering_results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    clustering_results_df["K"],
    clustering_results_df["WCSS"],
    marker="o"
)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("WCSS")
ax.set_title("Elbow Method for Selecting K")
ax.set_xticks(list(K_VALUES))
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    clustering_results_df["K"],
    clustering_results_df["Silhouette"],
    marker="o"
)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score by K")
ax.set_xticks(list(K_VALUES))
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

best_silhouette_k = int(
    clustering_results_df.loc[
        clustering_results_df["Silhouette"].idxmax(), "K"
    ]
)

print("Highest silhouette score occurs at K =", best_silhouette_k)
print("Select the final K after considering both the elbow pattern and silhouette results.")

## Final K-Means++ Model

The final K should be selected after inspecting the Elbow and Silhouette results. The default below uses the K with the highest silhouette score, but it can be changed manually if the elbow pattern and cluster interpretability support another K.

In [ ]:
# Default: K with the highest silhouette score.
# Change FINAL_K manually after inspecting the plots if required.

FINAL_K = int(
    clustering_results_df.loc[
        clustering_results_df["Silhouette"].idxmax(), "K"
    ]
)

print("Selected K:", FINAL_K)

In [ ]:
final_kmeans = KMeans(
    n_clusters=FINAL_K,
    init="k-means++",
    n_init=20,
    max_iter=300,
    random_state=42,
)

train_cluster_labels = final_kmeans.fit_predict(X_train_cluster)

print("Final clustering completed.")
print("Training cluster distribution:")
print(pd.Series(train_cluster_labels).value_counts().sort_index())
print("\nFinal WCSS:", f"{final_kmeans.inertia_:,.2f}")

## Optional Mini-Batch K-Means Comparison

The lecturer material also introduces **Mini-Batch K-Means** as a more computationally efficient variant of K-Means. It updates centroids using small batches rather than the complete dataset at every iteration.

This is included as a computational comparison only. The standard K-Means++ solution remains the primary clustering result unless the experiment provides a reason to use Mini-Batch K-Means.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import time

start = time.perf_counter()

minibatch_kmeans = MiniBatchKMeans(
    n_clusters=FINAL_K,
    init="k-means++",
    n_init=10,
    batch_size=1024,
    random_state=RANDOM_STATE,
    max_iter=300
)

minibatch_labels = minibatch_kmeans.fit_predict(X_train_cluster)

mb_time = time.perf_counter() - start

print("Mini-Batch K-Means comparison")
print("K:", FINAL_K)
print("Inertia:", f"{minibatch_kmeans.inertia_:,.2f}")
print("Training time (sec):", f"{mb_time:.3f}")

print("\nStandard K-Means inertia:", f"{final_kmeans.inertia_:,.2f}")


## Cluster Profiling

Cluster profiles are summarized using the selected encoded features. Cluster numbers are arbitrary; the actual archetype descriptions should be based on the observed feature profiles.

In [ ]:
profile_df = pd.DataFrame(
    X_train_cluster,
    columns=selected_feature_names
)
profile_df["Cluster"] = train_cluster_labels

cluster_sizes = (
    profile_df["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Count")
    .to_frame()
)
cluster_sizes["Percentage"] = (
    cluster_sizes["Count"] / len(profile_df) * 100
)

print("Cluster sizes:")
display(cluster_sizes)

cluster_means = profile_df.groupby("Cluster").mean(numeric_only=True)
display(cluster_means.round(3))

In [ ]:
# Strongest distinguishing encoded features for each cluster.
global_means = profile_df.drop(columns="Cluster").mean()
cluster_z = cluster_means.subtract(global_means, axis=1)

for cluster_id in cluster_z.index:
    strongest = (
        cluster_z.loc[cluster_id]
        .abs()
        .sort_values(ascending=False)
        .head(15)
        .index
    )

    print(f"\nCluster {cluster_id} — strongest distinguishing features")
    display(
        cluster_z.loc[cluster_id, strongest]
        .sort_values(key=np.abs, ascending=False)
        .to_frame("Standardized Difference")
    )

## Relationship Between Crash Archetypes and Severity

Severity is **not** used to create the clusters. It is used only after clustering to describe how the discovered crash groups relate to the target outcome.

In [ ]:
severity_names = {
    0: "Fatal",
    1: "Other Injury",
    2: "Serious Injury"
}

severity_profile = pd.crosstab(
    train_cluster_labels,
    pd.Series(y_train).map(severity_names),
    normalize="index"
) * 100

severity_counts = pd.crosstab(
    train_cluster_labels,
    pd.Series(y_train).map(severity_names)
)

print("Severity distribution within each cluster (%)")
display(severity_profile.round(2))

print("Severity counts within each cluster")
display(severity_counts)

In [ ]:
ax = severity_profile.plot(
    kind="bar",
    figsize=(10, 5),
    stacked=True
)
ax.set_xlabel("Cluster")
ax.set_ylabel("Percentage of Cluster")
ax.set_title("Severity Composition of Crash Archetypes")
ax.legend(title="Severity")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Assign Archetypes to the Unseen Test Set

The K-Means model is fitted only on training data. Test crashes are assigned to the nearest learned centroid without refitting the clustering model.

In [ ]:
X_test_cluster = X_test_selected.toarray().astype(np.float32)
test_cluster_labels = final_kmeans.predict(X_test_cluster)

print("Test cluster distribution:")
print(pd.Series(test_cluster_labels).value_counts().sort_index())

## Save Cluster Assignments

These labels will be used in the next supervised-learning stage as a new engineered feature.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

np.save(
    os.path.join(DATA_DIR, "crash_cluster_train.npy"),
    train_cluster_labels
)

np.save(
    os.path.join(DATA_DIR, "crash_cluster_test.npy"),
    test_cluster_labels
)

np.save(
    os.path.join(DATA_DIR, "crash_cluster_centers.npy"),
    final_kmeans.cluster_centers_
)

clustering_results_df.to_csv(
    os.path.join(DATA_DIR, "kmeans_k_selection_results.csv"),
    index=False
)

cluster_sizes.to_csv(
    os.path.join(DATA_DIR, "crash_cluster_summary.csv")
)

print("Saved:")
print("- crash_cluster_train.npy")
print("- crash_cluster_test.npy")
print("- crash_cluster_centers.npy")
print("- kmeans_k_selection_results.csv")
print("- crash_cluster_summary.csv")

## Key Interpretation Notes

- Cluster numbers are arbitrary labels; Cluster 0 is not inherently lower or higher risk than another cluster.
- K-Means discovers groups from the input features without using `SEVERITY`.
- Severity percentages are calculated **after** clustering only to interpret the crash archetypes.
- The final K is supported by the WCSS/Elbow and Silhouette analyses rather than by the severity distribution.
- If Mini-Batch K-Means is included, it is treated as an efficiency comparison, not automatically as the final clustering method.
